# 🧠 PagedAttention を理解する：概念図 ＋ vLLM 実測デモ

| パート | 内容 | GPU 必要？ |
|---|---|---|
| **Part 1** | PagedAttention の概念を matplotlib で可視化 | ❌ 不要 |
| **Part 2** | vLLM を実際に動かしてスループット・メモリを実測 | ✅ 必要 |

> **準備：** ランタイム → ランタイムのタイプを変更 → **GPU (T4)** を選択してください。

## ⚙️ インストール

> 完了後は **ランタイム → セッションを再起動** してから Part 1 に進んでください。

In [ ]:
# 日本語フォントのインストール
!apt-get install -y fonts-noto-cjk -q

# vLLM と依存パッケージ
!pip install vllm==0.6.6 numpy==1.26.4 protobuf==4.25.3 transformers==4.46.3 matplotlib -q

print("✅ インストール完了！ランタイムを再起動してください。")
print("   メニュー: ランタイム → セッションを再起動")

## ⚙️ 日本語フォントの設定

> ランタイム再起動後、**最初に必ずこのセルを実行**してください。

In [ ]:
import matplotlib
import matplotlib.font_manager as fm

# フォントキャッシュをリセットして Noto CJK を登録
fm._load_fontmanager(try_read_cache=False)
font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
fm.fontManager.addfont(font_path)
matplotlib.rcParams['font.family'] = 'Noto Sans CJK JP'
matplotlib.rcParams['axes.unicode_minus'] = False

# 確認
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 1))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')
ax.text(0.5, 0.5, '日本語フォント設定完了 ✅',
        ha='center', va='center', color='#4ade80', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

---
# Part 1：概念の可視化

## 1-1. 従来方式 vs PagedAttention のメモリ管理

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#0f1117')

COLORS = {
    'wasted': '#f87171',
    'free':   '#1e293b',
    'border': '#334155',
    'bg':     '#0f1117',
    'text':   '#e2e8f0',
}
requests = [
    {'label': 'A', 'used': 3, 'reserved': 8, 'color': '#818cf8'},
    {'label': 'B', 'used': 6, 'reserved': 8, 'color': '#38bdf8'},
    {'label': 'C', 'used': 2, 'reserved': 8, 'color': '#fb923c'},
]
total_slots = 32
slot_w, slot_h = 0.9, 0.8

# --- 左：従来方式 ---
ax = axes[0]
ax.set_facecolor(COLORS['bg'])
ax.set_title('従来方式（連続メモリ確保）', color=COLORS['text'], fontsize=13, fontweight='bold', pad=12)
slot_idx = 0
for req in requests:
    for i in range(req['reserved']):
        row, col = slot_idx // 8, slot_idx % 8
        x, y = col*(slot_w+0.1), row*(slot_h+0.1)
        color = req['color'] if i < req['used'] else COLORS['wasted']
        alpha = 0.9 if i < req['used'] else 0.35
        ax.add_patch(mpatches.FancyBboxPatch((x, y), slot_w, slot_h,
            boxstyle='round,pad=0.05', facecolor=color,
            edgecolor='#475569', linewidth=0.8, alpha=alpha))
        if i == 0:
            ax.text(x+slot_w/2, y+slot_h/2, req['label'],
                    ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        slot_idx += 1
for i in range(total_slots - slot_idx):
    row, col = slot_idx // 8, slot_idx % 8
    x, y = col*(slot_w+0.1), row*(slot_h+0.1)
    ax.add_patch(mpatches.FancyBboxPatch((x, y), slot_w, slot_h,
        boxstyle='round,pad=0.05', facecolor=COLORS['free'],
        edgecolor=COLORS['border'], linewidth=0.8))
    slot_idx += 1

total_used     = sum(r['used']     for r in requests)
total_reserved = sum(r['reserved'] for r in requests)
wasted = total_reserved - total_used
eff    = total_used / total_reserved * 100
ax.text(4, -0.9,
        f'使用: {total_used}  予約: {total_reserved}  無駄: {wasted}  効率: {eff:.0f}%',
        ha='center', fontsize=10, color='#f87171', fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='#818cf8', label='Request A'),
    mpatches.Patch(color='#38bdf8', label='Request B'),
    mpatches.Patch(color='#fb923c', label='Request C'),
    mpatches.Patch(color=COLORS['wasted'], alpha=0.35, label='予約済み（未使用・無駄）'),
    mpatches.Patch(color=COLORS['free'],   label='空きスロット'),
], loc='upper right', facecolor='#1e293b', edgecolor='#475569',
   labelcolor=COLORS['text'], fontsize=8)
ax.set_xlim(-0.2, 8); ax.set_ylim(-1.3, 4); ax.axis('off')

# --- 右：PagedAttention ---
ax2 = axes[1]
ax2.set_facecolor(COLORS['bg'])
ax2.set_title('PagedAttention（ブロック単位の柔軟な管理）',
              color=COLORS['text'], fontsize=13, fontweight='bold', pad=12)
paged = (
    [{'color': '#818cf8', 'label': 'A'}] * 3 +
    [{'color': '#38bdf8', 'label': 'B'}] * 6 +
    [{'color': '#fb923c', 'label': 'C'}] * 2 +
    [{'color': COLORS['free'], 'label': ''}] * (total_slots - 11)
)
for idx, block in enumerate(paged):
    row, col = idx // 8, idx % 8
    x, y = col*(slot_w+0.1), row*(slot_h+0.1)
    is_used = block['label'] != ''
    ax2.add_patch(mpatches.FancyBboxPatch((x, y), slot_w, slot_h,
        boxstyle='round,pad=0.05', facecolor=block['color'],
        edgecolor='#94a3b8' if is_used else COLORS['border'],
        linewidth=1.5 if is_used else 0.8))
    if is_used:
        ax2.text(x+slot_w/2, y+slot_h/2, block['label'],
                 ha='center', va='center', fontsize=8, color='white', fontweight='bold')

paged_eff = total_used / total_slots * 100
ax2.text(4, -0.9,
         f'使用: {total_used}  予約: {total_used}  無駄: 0  効率: {paged_eff:.0f}%',
         ha='center', fontsize=10, color='#4ade80', fontweight='bold')
ax2.legend(handles=[
    mpatches.Patch(color='#818cf8', label='Request A'),
    mpatches.Patch(color='#38bdf8', label='Request B'),
    mpatches.Patch(color='#fb923c', label='Request C'),
    mpatches.Patch(color=COLORS['free'], label='空きスロット（次のリクエストに使える）'),
], loc='upper right', facecolor='#1e293b', edgecolor='#475569',
   labelcolor=COLORS['text'], fontsize=8)
ax2.set_xlim(-0.2, 8); ax2.set_ylim(-1.3, 4); ax2.axis('off')

fig.suptitle('KV キャッシュのメモリ管理比較',
             color=COLORS['text'], fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 1-2. トークン生成によるブロック割り当てアニメーション

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

BLOCK_SIZE = 4
NUM_BLOCKS = 12
MAX_TOKENS = 20
AC = {'bg':'#0f1117','free':'#1e293b','used':'#4ade80','active':'#facc15','text':'#e2e8f0'}

fig2, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(14, 8),
                                       gridspec_kw={'height_ratios': [2, 1]})
fig2.patch.set_facecolor(AC['bg'])

def draw_frame(step):
    ax_top.cla(); ax_bot.cla()
    ax_top.set_facecolor(AC['bg']); ax_bot.set_facecolor(AC['bg'])

    blocks_needed   = max(1, (step + BLOCK_SIZE - 1) // BLOCK_SIZE)
    tokens_in_last  = step % BLOCK_SIZE if step % BLOCK_SIZE != 0 else (BLOCK_SIZE if step > 0 else 0)

    ax_top.set_title(
        f'物理ブロック（GPU 上の KV キャッシュ）  生成トークン数: {step} / {MAX_TOKENS}',
        color=AC['text'], fontsize=12, fontweight='bold')

    bw, bh, gap, cols = 1.8, 1.2, 0.3, 6
    for i in range(NUM_BLOCKS):
        col2, row2 = i % cols, i // cols
        x, y = col2*(bw+gap), -row2*(bh+gap)
        if step == 0:
            fc, lbl = AC['free'], f'Block {i}\n空き'
        elif i < blocks_needed - 1:
            fc, lbl = AC['used'], f'Block {i}\n満杯'
        elif i == blocks_needed - 1:
            fc, lbl = AC['active'], f'Block {i}\n書込中 {tokens_in_last}/{BLOCK_SIZE}'
        else:
            fc, lbl = AC['free'], f'Block {i}\n空き'
        lc = '#0f1117' if fc != AC['free'] else '#94a3b8'
        ew = 2 if (step > 0 and i == blocks_needed - 1) else 1
        ax_top.add_patch(mpatches.FancyBboxPatch((x, y), bw, bh,
            boxstyle='round,pad=0.08', facecolor=fc,
            edgecolor='#94a3b8' if ew == 2 else '#475569', linewidth=ew))
        ax_top.text(x+bw/2, y+bh/2, lbl, ha='center', va='center',
                    fontsize=8.5, color=lc)

    ax_top.legend(handles=[
        mpatches.Patch(color=AC['used'],   label='使用済み（満杯）'),
        mpatches.Patch(color=AC['active'], label='アクティブ（書込中）'),
        mpatches.Patch(color=AC['free'],   label='空きブロック'),
    ], loc='upper right', facecolor='#1e293b', edgecolor='#475569',
       labelcolor=AC['text'], fontsize=8)
    ax_top.set_xlim(-0.3, cols*(bw+gap))
    ax_top.set_ylim(-2*(bh+gap)-0.3, bh+0.3)
    ax_top.axis('off')

    total_cap      = NUM_BLOCKS * BLOCK_SIZE
    trad_reserved  = blocks_needed * BLOCK_SIZE
    trad_wasted    = trad_reserved - step
    trad_util      = step / trad_reserved * 100 if trad_reserved > 0 else 0
    paged_util     = step / total_cap * 100

    for bi, (label, vals, cols_c, note, nc) in enumerate([
        ('従来方式',
         [step, trad_wasted, total_cap - trad_reserved],
         [AC['used'], '#f87171', AC['free']],
         f'{trad_util:.0f}% 効率', '#f87171'),
        ('PagedAttention',
         [step, 0, total_cap - step],
         [AC['used'], AC['active'], AC['free']],
         f'{paged_util:.0f}% 効率', '#4ade80'),
    ]):
        left = 0
        for v, c in zip(vals, cols_c):
            ax_bot.barh(bi, v, left=left, color=c, height=0.5,
                        edgecolor='#0f1117', linewidth=0.5)
            left += v
        ax_bot.text(-0.5, bi, label, ha='right', va='center',
                    color=AC['text'], fontsize=9)
        ax_bot.text(total_cap+0.5, bi, note, ha='left', va='center',
                    color=nc, fontsize=9, fontweight='bold')

    ax_bot.set_xlim(-8, total_cap+10)
    ax_bot.set_ylim(-0.5, 1.5)
    ax_bot.set_xlabel('トークン数', color=AC['text'])
    ax_bot.tick_params(colors=AC['text'])
    ax_bot.spines[:].set_color('#334155')
    ax_bot.set_yticks([])
    ax_bot.set_title('メモリ使用効率の比較', color=AC['text'], fontsize=10)
    fig2.tight_layout()

ani = animation.FuncAnimation(fig2, draw_frame,
                               frames=list(range(0, MAX_TOKENS+1)),
                               interval=600, repeat=True)
plt.close()
HTML(ani.to_jshtml())

## 1-3. プレフィックス共有（Copy-on-Write）の図解

In [ ]:
fig3, axes3 = plt.subplots(1, 2, figsize=(16, 7))
fig3.patch.set_facecolor('#0f1117')
bw2, bh2, gap_x2, gap_y2 = 2.0, 0.9, 0.4, 0.5

configs = [
    (axes3[0],
     '従来方式（プロンプトを各リクエストがコピー）',
     [(0,0,'#818cf8','Prompt Block 0'),
      (0,1,'#818cf8','Prompt Block 1'),
      (0,2,'#818cf8','Prompt Block 2'),
      (0,3,'#4ade80','Gen A Block 0'),
      (2,0,'#818cf8','Prompt Block 0\n(コピー)'),
      (2,1,'#818cf8','Prompt Block 1\n(コピー)'),
      (2,2,'#818cf8','Prompt Block 2\n(コピー)'),
      (2,3,'#f472b6','Gen B Block 0')],
     'メモリ: プロンプト 3x2 + 生成 2 = 8 ブロック', '#f87171', False),
    (axes3[1],
     'PagedAttention（プロンプトブロックを共有）',
     [(1,0,'#818cf8','共有 Prompt\nBlock 0'),
      (1,1,'#818cf8','共有 Prompt\nBlock 1'),
      (1,2,'#818cf8','共有 Prompt\nBlock 2'),
      (0,4,'#4ade80','Gen A Block 0'),
      (2,4,'#f472b6','Gen B Block 0')],
     'メモリ: プロンプト 3(共有) + 生成 2 = 5 ブロック → 37% 削減', '#4ade80', True),
]

for ax3, title, layout, stats, stat_color, draw_arrows in configs:
    ax3.set_facecolor('#0f1117')
    ax3.set_title(title, color='#e2e8f0', fontsize=11, fontweight='bold', pad=15)
    positions3 = {}
    for (col3, row3, color3, label3) in layout:
        x3 = col3*(bw2+gap_x2)
        y3 = -row3*(bh2+gap_y2)
        positions3[(col3, row3)] = (x3+bw2/2, y3+bh2/2)
        ax3.add_patch(mpatches.FancyBboxPatch((x3, y3), bw2, bh2,
            boxstyle='round,pad=0.08', facecolor=color3,
            edgecolor='#94a3b8', linewidth=1.5))
        ax3.text(x3+bw2/2, y3+bh2/2, label3, ha='center', va='center',
                 fontsize=8, color='#0f1117', fontweight='bold')
    if draw_arrows:
        for src in [(1,0),(1,1),(1,2)]:
            for dst in [(0,4),(2,4)]:
                if src in positions3 and dst in positions3:
                    sx3, sy3 = positions3[src]
                    dx3, dy3 = positions3[dst]
                    ax3.annotate('', xy=(dx3, dy3), xytext=(sx3, sy3),
                                 arrowprops=dict(arrowstyle='->',
                                     color='#64748b', lw=1.2,
                                     connectionstyle='arc3,rad=0.2'))
    ax3.text(2.3, -5.2, stats, ha='center',
             color=stat_color, fontsize=9, fontweight='bold')
    ax3.set_xlim(-0.5, 4*(bw2+gap_x2))
    ax3.set_ylim(-5.5*(bh2+gap_y2), bh2+1.0)
    ax3.axis('off')

fig3.suptitle('プレフィックス共有（Copy-on-Write）',
              color='#e2e8f0', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 2：vLLM 実測デモ

## 2-1. モデルのロード

In [ ]:
from vllm import LLM, SamplingParams
import torch, time

print("モデルをロード中...")
llm = LLM(
    model="facebook/opt-125m",
    dtype="float16",
    enforce_eager=True,
    gpu_memory_utilization=0.5,
)
print("✅ モデルのロード完了")

## 2-2. 並列リクエスト数を変えてスループットを実測

In [ ]:
def benchmark(llm, num_requests, max_tokens=64):
    prompts = [
        f"Tell me about artificial intelligence in detail. Request {i}:"
        for i in range(num_requests)
    ]
    params = SamplingParams(temperature=0.0, max_tokens=max_tokens)
    torch.cuda.synchronize()
    start = time.perf_counter()
    outputs = llm.generate(prompts, params)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    return total_tokens / elapsed, elapsed, total_tokens, torch.cuda.memory_allocated() / 1024**3

batch_sizes = [1, 2, 4, 8, 16]
results = []

print(f"{'並列数':>6}  {'スループット':>14}  {'時間':>8}  {'総トークン':>10}  {'GPU メモリ':>10}")
print("-" * 60)
for n in batch_sizes:
    tp, elapsed, total_tok, mem = benchmark(llm, n)
    results.append({'n': n, 'throughput': tp, 'elapsed': elapsed, 'tokens': total_tok, 'mem': mem})
    print(f"{n:>6}  {tp:>12.1f}/s  {elapsed:>6.2f}s  {total_tok:>10}  {mem:>8.2f} GB")

## 2-3. 実測結果の可視化

In [ ]:
fig4, axes4 = plt.subplots(1, 3, figsize=(18, 5))
fig4.patch.set_facecolor('#0f1117')

ns          = [r['n']          for r in results]
throughputs = [r['throughput'] for r in results]
latencies   = [r['elapsed']    for r in results]
mems        = [r['mem']        for r in results]

for ax4, x, y, color, title, xlabel, ylabel in [
    (axes4[0], ns, throughputs, '#4ade80', 'スループット vs 並列数', '並列リクエスト数', 'tokens/s'),
    (axes4[1], ns, latencies,   '#facc15', 'レイテンシ vs 並列数',   '並列リクエスト数', '秒 (s)'),
    (axes4[2], ns, mems,        '#f472b6', 'GPU メモリ vs 並列数',   '並列リクエスト数', 'GB'),
]:
    ax4.set_facecolor('#0f1117')
    ax4.plot(x, y, color=color, marker='o', linewidth=2.5,
             markersize=8, markerfacecolor='white',
             markeredgecolor=color, markeredgewidth=2)
    for xi, yi in zip(x, y):
        ax4.annotate(f'{yi:.1f}', (xi, yi),
                     textcoords='offset points', xytext=(0, 10),
                     ha='center', color='#e2e8f0', fontsize=9)
    ax4.set_title(title,   color='#e2e8f0', fontsize=11, fontweight='bold')
    ax4.set_xlabel(xlabel, color='#94a3b8')
    ax4.set_ylabel(ylabel, color='#94a3b8')
    ax4.tick_params(colors='#94a3b8')
    ax4.spines[:].set_color('#334155')
    ax4.set_xticks(x)
    ax4.grid(True, color='#1e293b', linewidth=0.8)

fig4.suptitle('vLLM 実測結果（facebook/opt-125m、T4 GPU）',
              color='#e2e8f0', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best  = max(results, key=lambda r: r['throughput'])
worst = min(results, key=lambda r: r['throughput'])
print(f"""
📊 実測サマリー
  最高スループット : {best['throughput']:.1f} tokens/s（並列 {best['n']} リクエスト）
  最低スループット : {worst['throughput']:.1f} tokens/s（並列 {worst['n']} リクエスト）
  スループット倍率 : {best['throughput']/worst['throughput']:.1f}x
""")

---
## 📌 まとめ

| | 従来方式 | PagedAttention (vLLM) |
|---|---|---|
| KV キャッシュ管理 | 連続メモリを事前確保 | ブロック単位で動的割り当て |
| メモリ効率 | 20〜40% が無駄になることも | ほぼ 100% 活用 |
| プロンプト共有 | 不可 | Copy-on-Write で共有可 |
| 並列リクエスト増加時 | メモリ不足になりやすい | 柔軟にスケール |
| スループット向上 | ベースライン | 最大 2〜3.5x |